In [ ]:
library(Seurat)
library(dplyr)
library(data.table)
library(ggplot2)

library(reticulate)
use_condaenv("skinR44Py312", required = TRUE)

source("~/Projects/heads/clustering.r")

In [ ]:
data_dir = "/gpfs/gibbs/pi/braun/zy325/scrcc/processed/seuratobj_2026/"

In [ ]:
obj = readRDS(file.path(data_dir,"scrcc_cd8t_annotated.rds"))

#obj = readRDS("scrcc_t_ilc2t_cd8t_clustered.rds")
#obj = subset(obj,lineage3.5 == "CD8T")

In [ ]:
obj = clustering(obj,keep_c_genes = F,
                harmony_theta = 3,vars.to.regress = c("nFeature_RNA","nCount_RNA","percent.mt"),
                plot_QC_metrics = F,
                group.by.vars = "batch_lab",dims = 1:20,resolution=1.2)

In [ ]:
#saveRDS(obj,file=file.path(data_dir,"processed","scrcc_cd8t_clustered.rds"))

In [ ]:
obj = readRDS(file.path(data_dir,"processed","scrcc_cd8t_clustered.rds"))

In [ ]:
DimPlot(obj, label=T) #+ NoLegend()

In [ ]:
DimPlot(obj,group.by = "batch_lab") #+ NoLegend()

In [ ]:
DimPlot(obj,group.by = "batch_seq_rna") #+ NoLegend()

In [ ]:
options(repr.plot.width=10,repr.plot.height=10)
obj %>% 
VlnPlot(fill.by = "ident",features = c(
    "CD3D","CD3E","CD3G","CD8A","CD8B",
    "FOXP3","IKZF2",
    "CXCL13","PDCD1","HAVCR2",
    "IFNG","MX1","ISG15",
    "TOX","ZNF683","ITGA1",
    "NR4A1","LMNA","DNAJB1","HSPA1A",
    "SLC4A10","KLRB1","IL7R",
    "FCGR3A","PRF1","GZMB"),pt.size = 0,stack = T,flip = T)

In [ ]:
options(repr.plot.width=15,repr.plot.height=10)
obj = NormalizeData(obj,assay = "AIR")
VlnPlot(obj,features = c("TRBC1","TRBC2","TRAC","TRDC","TRDV2","TRGV9","TRAV1-2","TRAV11"),assay = "AIR",pt.size = 0,stack = T,flip = T)

In [ ]:
table(obj$`RNA_snn_res.1`[obj$anno_cd8t == "CD8Tex_NMF3"]) %>% sort(decreasing = TRUE)
table(obj$`RNA_snn_res.1`[obj$anno_cd8t == "CD8Tstr_HSPA6"]) %>% sort(decreasing = TRUE)
table(obj$`RNA_snn_res.1`[obj$anno_cd8t == "CD8Tn_CCR7"]) %>% sort(decreasing = TRUE)

In [ ]:
### Combining expression markers & TCR genes ###

obj$lineage4 = case_when(
    obj$`RNA_snn_res.0.5` %in% 
    obj$`RNA_snn_res.0.5` %in% 
    obj$`RNA_snn_res.0.5` %in% 
    TRUE~"contamination")

In [ ]:
options(repr.plot.width=8,repr.plot.height=7)
DimPlot(obj,group.by = "wTCR") #+ NoLegend()

In [ ]:
options(repr.plot.width=7,repr.plot.height=7)
DimPlot(obj,cells.highlight = obj$name[obj$anno_cd8t == "CD8Tex_NMF3"],raster = T) + NoLegend()

In [ ]:
options(repr.plot.width=8,repr.plot.height=7)
DimPlot(obj,group.by = "lineage4") #+ NoLegend()

In [ ]:
#saveRDS(subset(obj,lineage4=="CD8T"),file=file.path(data_dir,"processed","scrcc_cd8t_manuscript.rds"))

In [ ]:
#saveRDS(subset(obj,`RNA_snn_res.0.5` %in% c(0,1,8,10,13)),file=file.path(data_dir,"processed","scrcc_tex_test.rds"))

In [ ]:
obj@meta.data %>% head

In [ ]:
obj$lineage5 = obj$lineage4
tex_id = which(obj$lineage5 == "CD8Tex_PDCD1")
obj$lineage5[tex_id] = obj$anno_tex[tex_id]
obj$lineage5[is.na(obj$lineage5)] = "Tex_ANT"

In [ ]:
### response ###

options(repr.plot.width=12,repr.plot.height=15)

CB = paste0("SCRCC",c('24', '05', '10', '50', '38', '42', '80', '03', '54', '64', '12', '29', '09', '21', '53', '59', '16'))
NCB = paste0("SCRCC",c("01", "02", "47", "45", "46", "55", "06", "40", "74", "11", "07"))

n_cd8t = obj@meta.data %>% group_by(sample_id2) %>% summarise(n_cells=n()) %>% filter(n_cells>=100) %>% .$sample_id2

obj@meta.data %>%
    filter(sample_id2 %in% n_cd8t) %>%
    group_by(sample_id2,lineage5) %>%
    summarise(n_cells = n(),.groups = "drop_last") %>%
    mutate(proportion=n_cells/sum(n_cells)) %>%
    mutate(
        response=case_when(sample_id2 %in% CB~"R",sample_id2 %in% NCB~"NR",TRUE~"UK")) %>%
    filter(response!="UK") %>%
    ggplot(aes(x = response,y = proportion,fill=response))+
    geom_boxplot(outlier.shape = NA)+
    geom_jitter()+
    stat_compare_means()+
    facet_wrap(.~lineage5,ncol = 4)+
    theme_bw()

In [ ]:
m = FindMarkers(obj,`ident.1` = 4,only.pos = T,logfc.threshold = .5)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
m = FindMarkers(obj,`ident.1` = 7,only.pos = T,logfc.threshold = .5)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
m = FindMarkers(obj,`ident.1` = 8,only.pos = T,logfc.threshold = .5)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
m = FindMarkers(obj,`ident.1` = 9,only.pos = T,logfc.threshold = .5)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
m = FindMarkers(obj,`ident.1` = 5,only.pos = T,logfc.threshold = .5)
m %>% filter(p_val_adj<0.05) %>% arrange(desc(avg_log2FC)) %>% filter(abs(pct.1-pct.2)>.08)

In [ ]:
library(ggsankey)
options(repr.plot.width=7, repr.plot.height=10)

df <- obj@meta.data %>%
    filter(lineage5 == "nmf3") %>%
    #filter(anno_cd8t == "CD8Tex_NMF3") %>%
    make_long(lineage5, anno4)

df <- df %>%
    group_by(x, node) %>%
    mutate(n = n()) %>%
    ungroup() %>%
    mutate(label_n = paste0(node, " (", n, ")"))

node_order <- df %>%
    distinct(node, n) %>%
    group_by(node) %>%
    summarise(n = max(n), .groups = "drop") %>%
    arrange(n) %>%
    pull(node)

df <- df %>%
    mutate(node = factor(node, levels = node_order),
           next_node = factor(next_node, levels = node_order))

df %>%
    ggplot(aes(x = x,
               next_x = next_x,
               node = node,
               next_node = next_node,
               fill = node,
               label = label_n)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3.5, color = 1, fill = "white", hjust = -0.1) +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
library(ggsankey)
options(repr.plot.width=7, repr.plot.height=10)

df <- obj@meta.data %>%
    filter(anno_cd8t == "CD8Tex_NMF3") %>%
    make_long(lineage5, anno_cd8t)

df <- df %>%
    group_by(x, node) %>%
    mutate(n = n()) %>%
    ungroup() %>%
    mutate(label_n = paste0(node, " (", n, ")"))

node_order <- df %>%
    distinct(node, n) %>%
    group_by(node) %>%
    summarise(n = max(n), .groups = "drop") %>%
    arrange(n) %>%
    pull(node)

df <- df %>%
    mutate(node = factor(node, levels = node_order),
           next_node = factor(next_node, levels = node_order))

df %>%
    ggplot(aes(x = x,
               next_x = next_x,
               node = node,
               next_node = next_node,
               fill = node,
               label = label_n)) +
    geom_sankey(flow.alpha = 0.5, node.color = 1) +
    geom_sankey_label(size = 3.5, color = 1, fill = "white", hjust = 1) +
    theme_sankey(base_size = 16) + NoLegend()

In [ ]:
tex = subset(obj, anno_tex %in% paste0("nmf",1:6))

In [ ]:
nmf_red = tex@reductions$pca

In [ ]:
nmf_loadings = tex@meta.data %>% select(paste0("loading_nmf",1:6))
colnames(nmf_loadings) = paste0("NMF_",1:6)

nmf_red@cell.embeddings = as.matrix(nmf_loadings)
nmf_red@key = "NMF_"

In [ ]:
tex@reductions$nmf = nmf_red

In [ ]:
tex = RunTSNE(tex,reduction = "nmf",dims = 1:6)

In [ ]:
DimPlot(tex,reduction = "tsne",group.by = "anno_tex")

In [ ]:
tex = RunUMAP(tex,reduction = "nmf",dims = 1:6)

In [ ]:
DimPlot(tex,reduction = "umap",group.by = "anno_tex")

In [ ]:
tex = FindNeighbors(tex,reduction = "nmf",dims = 1:6)

In [ ]:
tex = FindClusters(tex,resolution = .3)

In [ ]:
DimPlot(tex,reduction = "tsne")#,group.by = "anno_tex")

In [ ]:
table(tex$`RNA_snn_res.0.3`)